# Notebook 01 — Exploratory Data Analysis

Overall data quality, distributions, and churn rate breakdowns.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from skills.utils.plotting import save_fig

sns.set_theme(style='whitegrid')


## Load Data

In [2]:
df = pd.read_csv('../Customer-Churn-Records.csv')
print(f"Shape: {df.shape}")
df.head()


Shape: (10000, 18)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


## Data Quality

In [3]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
assert df.isnull().sum().sum() == 0, "Unexpected nulls found"
print("\nNo missing values.")


=== Data Types ===
RowNumber               int64
CustomerId              int64
Surname                object
CreditScore             int64
Geography              object
Gender                 object
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Card Type              object
Point Earned            int64
dtype: object

=== Missing Values ===
RowNumber             0
CustomerId            0
Surname               0
CreditScore           0
Geography             0
Gender                0
Age                   0
Tenure                0
Balance               0
NumOfProducts         0
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
Exited                0
Complain              0
Satisfaction Score    0
Card Type          

## Overall Churn Rate

In [4]:
churn_rate = df['Exited'].mean()
n_churned = df['Exited'].sum()
print(f"Overall churn rate: {churn_rate:.1%}  ({n_churned} of {len(df)} customers)")


Overall churn rate: 20.4%  (2038 of 10000 customers)


## Feature Distributions

In [5]:
features = ['Age', 'Balance', 'CreditScore', 'Tenure', 'Point Earned', 'EstimatedSalary']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), features):
    df[feat].hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(feat)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, '01_distributions.png', output_dir='../outputs/figures')
plt.close(fig)


## Correlation Heatmap

In [6]:
numeric_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
                'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited',
                'Complain', 'Satisfaction Score', 'Point Earned']
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df[numeric_cols].corr(numeric_only=True), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
save_fig(fig, '01_correlation_heatmap.png', output_dir='../outputs/figures')
plt.close(fig)


## Churn Rate by Categorical Features

In [7]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, ['Gender', 'IsActiveMember', 'NumOfProducts', 'HasCrCard']):
    churn_by = df.groupby(col)['Exited'].mean() * 100
    churn_by.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('Churn Rate by Categorical Features', fontsize=14)
plt.tight_layout()
save_fig(fig, '01_churn_by_categorical.png', output_dir='../outputs/figures')
plt.close(fig)
